# QUELL — Step 02: Veri inceleme (schema + class distribution)

Uc dataseti okumadan preprocessing yazmiyoruz. Bu notebook her biri icin:
columns, dtype, label column, **class distribution**, eksik/NaN, rows sayisi cikarir
ve `results/schema_report.json`'a yazar (reproducibility).

- CICIoT2023 (13 GB) bellege SIGMAZ → parca parca (chunk) okunur.
- Edge-IIoTset ML CSV kucuk → tam okunur.
- N-BaIoT → label file adindan/klasorden gelir (tek label column yok).

**JupyterHub'da hucre hucre calistir.** Sonuc JSON'unu bana getir → Faz 1 (split + baseline).

In [ ]:
import os, json, glob
from pathlib import Path
import pandas as pd

ROOT = Path.home() / "quell-edge-llm-ids"
RAW  = ROOT / "data" / "raw"
(ROOT / "results").mkdir(parents=True, exist_ok=True)
report = {}

LABEL_CANDS = ["label","attack","attack_type","attack_label","type","class","category","marker"]
def guess_label(cols):
    low = {c.lower(): c for c in cols}
    for k in LABEL_CANDS:
        if k in low: return low[k]
    # first column containing 'label'/'attack'/'class'
    for c in cols:
        if any(k in c.lower() for k in ("label","attack","class","categ")): return c
    return None
print("RAW:", RAW)

## CICIoT2023 (chunked)

In [ ]:
csvs = glob.glob(str(RAW/"ciciot2023"/"**/*.csv"), recursive=True)
csvs = [c for c in csvs if os.path.getsize(c) > 1_000_000]   # kucuk artiklari ele
print("files:", [os.path.basename(c) for c in csvs])
f = max(csvs, key=os.path.getsize)   # largest = merged csv
head = pd.read_csv(f, nrows=5, low_memory=False)
cols = list(head.columns); lab = guess_label(cols)
print("number of columns:", len(cols)); print("label column:", lab)
print("first columns:", cols[:10], "...")

# Reading the whole file as a single column (label) to get the REAL class distribution + row count
from collections import Counter
dist = Counter(); nrows = 0
if lab:
    for ch in pd.read_csv(f, usecols=[lab], chunksize=1_000_000, low_memory=False):
        dist.update(ch[lab].astype(str).value_counts().to_dict())
        nrows += len(ch)
        print(f"  ...rows read: {nrows:,}", end="\r")
print(f"\nTotal rows: {nrows:,}")
dist = dict(sorted(dist.items(), key=lambda x:-x[1]))
print("class distribution:"); [print(f"  {k}: {v:,}") for k,v in dist.items()]
report["ciciot2023"] = {"file": os.path.relpath(f,RAW), "n_rows": nrows, "n_cols": len(cols),
                         "label_col": lab, "n_classes": len(dist), "class_dist": dist,
                         "columns": cols}

## Edge-IIoTset (ML-EdgeIIoT-dataset.csv)

In [ ]:
cands = glob.glob(str(RAW/"edge_iiotset"/"**"/"ML-EdgeIIoT-dataset.csv"), recursive=True)
if not cands:
    cands = glob.glob(str(RAW/"edge_iiotset"/"**"/"DNN-EdgeIIoT-dataset.csv"), recursive=True)
f = cands[0]; print("kullanilan:", os.path.relpath(f,RAW))
df = pd.read_csv(f, low_memory=False)
print("sekil:", df.shape)
labs = [c for c in df.columns if c.lower() in ("attack_type","attack_label","label")]
print("possible label columns:", labs)
info = {"file": os.path.relpath(f,RAW), "n_rows": len(df), "n_cols": df.shape[1],
        "columns": list(df.columns), "label_cols": labs,
        "n_missing_total": int(df.isna().sum().sum())}
for lc in labs:
    d = df[lc].astype(str).value_counts().to_dict()
    info[f"dist__{lc}"] = d
    print(f"\n[{lc}] {len(d)} class:"); [print(f"  {k}: {v:,}") for k,v in list(d.items())[:20]]
report["edge_iiotset"] = info

## N-BaIoT (label file adindan)

In [ ]:
files = glob.glob(str(RAW/"nbaiot"/"**/*.csv"), recursive=True)
print("csv sayisi:", len(files))
def nbaiot_class(path):
    p = path.lower()
    if "benign" in p: return "benign"
    for k in ("combo","junk","scan","tcp","udp","ack","syn","udpplain"):
        if k in p: return ("mirai_" if "mirai" in p else "gafgyt_") + k
    return "unknown"
from collections import Counter
dist = Counter(); ncols=None; devices=set()
for fp in files:
    cls = nbaiot_class(fp)
    n = sum(len(ch) for ch in pd.read_csv(fp, usecols=[0], chunksize=500_000))
    dist[cls]+=n
    if ncols is None:
        ncols = pd.read_csv(fp, nrows=2).shape[1]
    # the device name is usually in the first folder/file part
    devices.add(os.path.basename(os.path.dirname(fp)) or os.path.basename(fp))
dist = dict(sorted(dist.items(), key=lambda x:-x[1]))
print("feature columns (unlabeled):", ncols)
print("class distribution:"); [print(f"  {k}: {v:,}") for k,v in dist.items()]
report["nbaiot"] = {"n_files": len(files), "n_feature_cols": ncols,
                    "n_classes": len(dist), "class_dist": dist,
                    "devices_sample": sorted(list(devices))[:15]}

## Rapor kaydet

In [ ]:
out = ROOT/"results"/"schema_report.json"
json.dump(report, open(out,"w"), indent=2, default=str, ensure_ascii=False)
print("yazildi ->", out)
print(json.dumps({k:{kk:vv for kk,vv in v.items() if kk!='columns' and not kk.startswith('dist__') and kk!='class_dist'}
                  for k,v in report.items()}, indent=2, ensure_ascii=False))